<a href="https://colab.research.google.com/github/usami0jp/ai2023-old/blob/main/rag_01c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

documents = [
    "架空のみどり市立あおぞら高校の創立は1987年で、校長は山田花子先生である。",
    "あおぞら高校の文化祭「そら祭」は毎年10月の第3土曜日と日曜日に開催される。",
    "あおぞら高校では、スマートフォンは昼休みと放課後のみ使用してよい。",
    "あおぞら高校の図書室は平日8時から18時まで開いており、1人5冊まで2週間借りられる。",
    "あおぞら高校の制服は紺のブレザーで、夏季は白のポロシャツも認められている。",
    "あおぞら高校の部活動で最も部員が多いのはバドミントン部で、約60人が所属している。",
]


embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_vecs = embedder.encode(
    ["passage: " + d for d in documents], normalize_embeddings=True
)


def retrieve(query, k=2):
    q_vec = embedder.encode(["query: " + query], normalize_embeddings=True)[0]
    scores = doc_vecs @ q_vec
    top = np.argsort(scores)[::-1][:k]
    return [(documents[i], float(scores[i])) for i in top]


generator = pipeline(
    "text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto"
)

def ask_llm(prompt):
    messages = [{"role": "user", "content": prompt}]
    out = generator(messages, max_new_tokens=200, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

def answer(query, use_rag=True, k=2):
    if not use_rag:
        return ask_llm(query)

    hits = retrieve(query, k)
    print("【検索された資料】")
    for doc, score in hits:
        print(f"  ({score:.3f}) {doc}")

    context = "\n".join(f"- {doc}" for doc, _ in hits)
    prompt = (
        "以下の資料だけを使って質問に答えてください。"
        "資料に答えがない場合は「資料にはありません」と答えてください。\n\n"
        f"資料:\n{context}\n\n質問: {query}"
    )
    return ask_llm(prompt)


if __name__ == "__main__":
    questions = [
        "あおぞら高校の文化祭はいつですか？",
        "図書室では何冊まで本を借りられますか？",
        "あおぞら高校の校歌の作曲者は誰ですか？",  # 資料にない質問
    ]
    for q in questions:
        print("=" * 60)
        print("質問:", q)
        print("\n▼ RAGなし")
        print(answer(q, use_rag=False))
        print("\n▼ RAGあり")
        print(answer(q, use_rag=True))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (8 outputs left out)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

KeyboardInterrupt: 